# PlayLogic GRPO Training

This notebook trains a small language model to emit valid football actions for the PlayLogic environment. Use a GPU runtime before running it.

In [4]:
# Setup
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/arjune4dev/PlayLogic.git"
REPO_DIR = pathlib.Path("/content/PlayLogic")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)

In [5]:
!pwd
!ls
!find . -name "requirements-training.txt"

/content/PlayLogic
Dockerfile    playlogic       README.md		test_ma.py  uv.lock
openenv.yaml  pyproject.toml  requirements.txt	training
./training/requirements-training.txt


In [6]:
import os, pathlib, subprocess, sys

os.chdir("/content/PlayLogic")
print("PWD:", pathlib.Path.cwd())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements-training.txt"])

os.environ["PYTHONPATH"] = "/content/PlayLogic"

print("Install done")

PWD: /content/PlayLogic
Install done


In [7]:
# Cell 1: Setup and Install (Fixed - No editable install)
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/arjune4dev/PlayLogic.git"
REPO_DIR = pathlib.Path("/content/PlayLogic")

# Clone the repository if it doesn't exist
if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

# Move into the repository
os.chdir(REPO_DIR)
print("Current directory:", pathlib.Path.cwd())

# Upgrade pip
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])

# Install only from requirements files (NO '-e .'!) and explicitly install trl
subprocess.check_call([
    sys.executable,
    "-m", "pip", "install", "-q",
    "-r", "requirements.txt",
    "-r", "training/requirements-training.txt",
    "trl"
])

# Tell Python where to find the local 'playlogic' package
os.environ["PYTHONPATH"] = "/content/PlayLogic"
print("PYTHONPATH set to:", os.environ["PYTHONPATH"])

print("✅ Install done – repo ready at", pathlib.Path.cwd())

Current directory: /content/PlayLogic
PYTHONPATH set to: /content/PlayLogic
✅ Install done – repo ready at /content/PlayLogic


In [8]:
import trl
print("trl installed:", trl.__version__)

trl installed: 1.2.0


In [9]:
from huggingface_hub import notebook_login, whoami
notebook_login()

In [10]:
from huggingface_hub import whoami
print(whoami()["name"])

ShikharNotFound


In [11]:
# Tiny proof run. Increase --max-steps after this succeeds.
!python training/train_grpo.py \
  --model-name Qwen/Qwen2-0.5B-Instruct \
  --output-dir ./grpo_football \
  --num-resets 4 \
  --max-prompts 32 \
  --max-steps 5 \
  --num-generations 2 \
  --per-device-train-batch-size 2 \
  --gradient-accumulation-steps 1

Environment smoke test OK
Rewards: {'A': np.float64(-0.09963037510887324), 'B': np.float64(0.0022099817073346364)}
Done: False
Dataset({
    features: ['prompt'],
    num_rows: 32
})
config.json: 100% 659/659 [00:00<00:00, 2.70MB/s]
tokenizer_config.json: 1.29kB [00:00, 3.00MB/s]
vocab.json: 2.78MB [00:00, 112MB/s]
merges.txt: 1.67MB [00:00, 119MB/s]
tokenizer.json: 7.03MB [00:00, 145MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 988M/988M [00:11<00:00, 87.8MB/s]
Loading weights: 100% 290/290 [00:01<00:00, 288.40it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 242/242 [00:00<00:00, 947kB/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
{'loss': '0', 'grad_norm': '0', 'learning_rate': '5e-06', 'num_tokens': '6

In [14]:
# Optional: push the trained model to Hugging Face Hub.
# 1. Create a token at https://huggingface.co/settings/tokens
# 2. Uncomment and run this cell.

import os
from huggingface_hub import notebook_login

notebook_login()

# Retrieve the token from the environment after login
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    print("Hugging Face token found in environment. Proceeding with training and push.")
    # Explicitly pass the HF_TOKEN to the subprocess command
    !HF_TOKEN="{hf_token}" python training/train_grpo.py \
      --model-name Qwen/Qwen2-0.5B-Instruct \
      --output-dir ./grpo_football_push \
      --num-resets 8 \
      --max-prompts 64 \
      --max-steps 50 \
      --num-generations 2 \
      --per-device-train-batch-size 2 \
      --gradient-accumulation-steps 1 \
      --push-to-hub \
      --hub-model-id ShikharNotFound/playlogic-football-grpo
else:
    print("Error: Hugging Face token not found after login. Please ensure you logged in successfully.")


Environment smoke test OK
Rewards: {'A': np.float64(-0.09963037510887324), 'B': np.float64(0.0022099817073346364)}
Done: False
Dataset({
    features: ['prompt'],
    num_rows: 64
})
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:03<00:00, 75.40it/s, Materializing param=model.norm.weight] 
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
{'loss': '0', 'grad_norm': '0', 'learning_rate': '5e-06', 'num_tokens': '362', 'completions/mean_length': '16', 'completions/min_length': '16', 'completions/max_length': '16', 'completions/clipped_ratio': '1', 'completions/mean_terminated_length': '0', 'completions/min_terminated_length': '0', 'completions/max_terminated_length': '0', 'rewards/football_action_reward/mean': '-1', 'rewards/football_ac